In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

print("Environment working!")

Environment working!


In [3]:
import pyodbc
import sklearn
import nltk

print("All libraries loaded successfully!")

All libraries loaded successfully!


In [4]:
import pyodbc

conn = pyodbc.connect(
    r"DRIVER={ODBC Driver 17 for SQL Server};"
    r"SERVER=localhost;"
    r"DATABASE=radiologyDB;"
    r"Trusted_Connection=yes;"
)

print("Connected successfully!")

Connected successfully!


In [5]:
query = "SELECT TOP 5 * FROM reports"

df = pd.read_sql(query, conn)

df.head()

C:\Users\HP\AppData\Local\Temp\ipykernel_15684\3551414036.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


,uid,MeSH,Problems,image,indication,comparison,findings,impression
0,1,normal,normal,Xray Chest PA and Lateral,Positive TB test,None.,The cardiac silhouette and mediastinum size ar...,Normal chest x-XXXX.
1,2,Cardiomegaly/borderline;Pulmonary Artery/enlarged,Cardiomegaly;Pulmonary Artery,"Chest, 2 views, frontal and lateral",Preop bariatric surgery.,None.,Borderline cardiomegaly. Midline sternotomy XX...,No acute pulmonary findings.
2,3,normal,normal,Xray Chest PA and Lateral,"rib pain after a XXXX, XXXX XXXX steps this XX...",NaN,NaN,"No displaced rib fractures, pneumothorax, or p..."
3,4,"Pulmonary Disease, Chronic Obstructive;Bullous...","Pulmonary Disease, Chronic Obstructive;Bullous...","PA and lateral views of the chest XXXX, XXXX a...",XXXX-year-old XXXX with XXXX.,None available,There are diffuse bilateral interstitial and a...,1. Bullous emphysema and interstitial fibrosis...
4,5,Osteophyte/thoracic vertebrae/multiple/small;T...,Osteophyte;Thickening;Lung,Xray Chest PA and Lateral,Chest and nasal congestion.,NaN,The cardiomediastinal silhouette and pulmonary...,No acute cardiopulmonary abnormality.


In [6]:
query = "SELECT * FROM reports"

df = pd.read_sql(query, conn)

print(df.shape)
df.head()

(3851, 8)


C:\Users\HP\AppData\Local\Temp\ipykernel_15684\1431509739.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


,uid,MeSH,Problems,image,indication,comparison,findings,impression
0,1,normal,normal,Xray Chest PA and Lateral,Positive TB test,None.,The cardiac silhouette and mediastinum size ar...,Normal chest x-XXXX.
1,2,Cardiomegaly/borderline;Pulmonary Artery/enlarged,Cardiomegaly;Pulmonary Artery,"Chest, 2 views, frontal and lateral",Preop bariatric surgery.,None.,Borderline cardiomegaly. Midline sternotomy XX...,No acute pulmonary findings.
2,3,normal,normal,Xray Chest PA and Lateral,"rib pain after a XXXX, XXXX XXXX steps this XX...",NaN,NaN,"No displaced rib fractures, pneumothorax, or p..."
3,4,"Pulmonary Disease, Chronic Obstructive;Bullous...","Pulmonary Disease, Chronic Obstructive;Bullous...","PA and lateral views of the chest XXXX, XXXX a...",XXXX-year-old XXXX with XXXX.,None available,There are diffuse bilateral interstitial and a...,1. Bullous emphysema and interstitial fibrosis...
4,5,Osteophyte/thoracic vertebrae/multiple/small;T...,Osteophyte;Thickening;Lung,Xray Chest PA and Lateral,Chest and nasal congestion.,NaN,The cardiomediastinal silhouette and pulmonary...,No acute cardiopulmonary abnormality.


In [7]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 3851 entries, 0 to 3850
Data columns (total 8 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   uid         3851 non-null   str  
 1   MeSH        3851 non-null   str  
 2   Problems    3851 non-null   str  
 3   image       3851 non-null   str  
 4   indication  3765 non-null   str  
 5   comparison  3252 non-null   str  
 6   findings    3337 non-null   str  
 7   impression  3820 non-null   str  
dtypes: str(8)
memory usage: 240.8 KB


In [8]:
df.isnull().sum()

uid             0
MeSH            0
Problems        0
image           0
indication     86
comparison    599
findings      514
impression     31
dtype: int64

In [9]:
missing_percent = (
    df.isnull().sum() / len(df)
) * 100

missing_percent.sort_values(ascending=False)

comparison    15.554401
findings      13.347183
indication     2.233186
impression     0.804986
image          0.000000
Problems       0.000000
MeSH           0.000000
uid            0.000000
dtype: float64

In [10]:
df["clinical_text"] = (
    df["impression"]
    .fillna(df["findings"])
    .fillna("")
)

df[["uid", "impression", "findings", "clinical_text"]].head(10)

,uid,impression,findings,clinical_text
0,1,Normal chest x-XXXX.,The cardiac silhouette and mediastinum size ar...,Normal chest x-XXXX.
1,2,No acute pulmonary findings.,Borderline cardiomegaly. Midline sternotomy XX...,No acute pulmonary findings.
2,3,"No displaced rib fractures, pneumothorax, or p...",NaN,"No displaced rib fractures, pneumothorax, or p..."
3,4,1. Bullous emphysema and interstitial fibrosis...,There are diffuse bilateral interstitial and a...,1. Bullous emphysema and interstitial fibrosis...
4,5,No acute cardiopulmonary abnormality.,The cardiomediastinal silhouette and pulmonary...,No acute cardiopulmonary abnormality.
5,6,No acute cardiopulmonary findings.,Heart size and mediastinal contour are within ...,No acute cardiopulmonary findings.
6,7,Basilar atelectasis. No confluent lobar consol...,The cardiac contours are normal. XXXX basilar ...,Basilar atelectasis. No confluent lobar consol...
7,8,No acute cardiopulmonary disease.,"The heart, pulmonary XXXX and mediastinum are ...",No acute cardiopulmonary disease.
8,9,Increased size of density in the left cardioph...,The XXXX examination consists of frontal and l...,Increased size of density in the left cardioph...
9,10,No acute cardiopulmonary process.,The cardiomediastinal silhouette is within nor...,No acute cardiopulmonary process.
